# **Collecte de la donnée**

### Modules et variables nécessaires

In [ ]:
#!pip install earthengine-api geemap setuptools
import ee
import geemap

PROJECT_ID = 'forestwatch-tg'

### Authentification et initialisation de EarthEngine

In [5]:
#!rm -rf ~/.config/earthengine/
#!earthengine authenticate
#!earthengine ls

ee.Authenticate()

True

In [6]:
try:
    ee.Initialize(project=PROJECT_ID)
except Exception as e:
    if 'not initialized' in str(e):
        ee.Authenticate(auth_mode="notebook")
        ee.Initialize(project=PROJECT_ID)
    else:
        raise e
    
print(f"✅ Google Earth Engine initialisé avec succès sur le projet : {PROJECT_ID}")

✅ Google Earth Engine initialisé avec succès sur le projet : forestwatch-tg


## Acquisition des images de Sentinel-2

##### **_Nature de la reception_**

| Type de donnée | Description | Résolution spatiale | Bandes disponibles |
|---------------|-------------|---------------------|--------------------|
| **Réflectance de surface** | Données corrigées des effets atmosphériques pour représenter la réflectance réelle des surfaces terrestres | 10 m, 20 m, 60 m | **B1** à **B12** (selon la résolution) |
| **Images multispectrales** | Captures dans différentes longueurs d’onde utilisées pour l’analyse du sol, de l’eau et de la végétation | 10 m, 20 m, 60 m | **B1** (Aérosols), **B2** (Bleu), **B3** (Vert), **B4** (Rouge), **B5-B7** (Rouge large), **B8** (NIR), **B8A** (NIR étroit), **B9** (Vapeur d'eau), **B10** (SWIR Cirrus), **B11-B12** (SWIR) |
| **Masques de classification** | Identification des nuages, de l’eau et des ombres pour améliorer l’analyse des images | 20 m, 60 m | Masques de qualité et classification des pixels |
| **Données atmosphériques** | Informations sur l’aérosol et la vapeur d’eau permettant des corrections atmosphériques | 60 m | **B1** (Aérosols), **B9** (Vapeur d’eau), **B10** (Cirrus) |

Plus d'infos : [Harmonized Sentinel-2 MSI: MultiSpectral Instrument, Level-2A (SR)](https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_SR_HARMONIZED?hl=fr#bands) <br> *https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_SR_HARMONIZED?hl=fr#bands*

### Acquisition

#### Filtre

In [7]:
def mask_s2_clouds_scl(image):
    scl = image.select('SCL')
    # Conserver uniquement : 4 (Végétation), 5 (Sols nus), 6 (Eau), 7 (Non classifié clean)
    # Exclure : 3 (Ombres de nuages), 8/9/10 (Nuages/Cirrus)
    mask = scl.remap([3, 8, 9, 10], [0, 0, 0, 0], 1)
    return image.updateMask(mask).divide(10000)

#### AOI et Extraction

In [9]:
# 1. Définition de la zone d'étude
# togo = ee.FeatureCollection("FAO/GAUL/2015/level0").filter(ee.Filter.eq('ADM0_NAME', 'Togo'))
# LSIB (Large Scale International Boundary Polygons)
togo = ee.FeatureCollection("USDOS/LSIB_SIMPLE/2017").filter(ee.Filter.eq('country_na', 'Togo'))

# 2. Récupérer la collection Sentinel-2 Harmonized
s2 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
    .filterBounds(togo.geometry()) \
    .filterDate('2025-01-01', '2025-12-31') \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)) \
    .map(mask_s2_clouds_scl) \
    .median() \
    .clip(togo.geometry())

In [ ]:
# Sélection des périmètres d'intérêt
#lome = ee.Geometry.Rectangle([1.1625, 6.0875, 1.2625, 6.1875])
#missaho = ee.Geometry.Rectangle([1.1575, 6.3700, 1.3575, 6.5700])
#tchamba = ee.Geometry.Rectangle([0.8000, 9.9600, 0.9500, 10.1100])
#dapaong = ee.Geometry.Rectangle([0.1833, 10.3150, 0.2833, 10.4150])
#atakora = ee.Geometry.Rectangle([0.5050, 10.2394, 0.7050, 10.4394])
#fazao_malfakassa = ee.Geometry.Rectangle([0.10, 9.05, 0.60, 9.35])
#abdoulaye = ee.Geometry.Rectangle([1.13, 8.34, 1.25, 8.47])
#alibi1 = ee.Geometry.Rectangle([1.10, 8.40, 1.20, 8.50])

# La zone d'étude (zone d'intérêt) est la superposition de ces périmètres
#study_area = lome.union(missaho).union(tchamba).union(dapaong).union(atakora).union(fazao_malfakassa).union(abdoulaye).union(alibi1)
#study_area = missaho.union(tchamba).union(atakora).union(fazao_malfakassa)

# Acquisition des images MSI_Level-2A(SR) de Sentinel-2 dans la période de décembre 2021
#collection = (
#    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
#    # Filtre sur une zone d'intérêt
#    .filterBounds(study_area)
#    # Filtre sur la période d'intérêt
#    .filterDate('2021-12-01', '2022-01-01')
#    # Filtre des images trop nuageuses.
#    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
#    # Masquage des pixels correspondants aux nuages et aux cirrus.
#    .map(mask_s2_clouds))

### Visualisation

In [15]:
# Visualisation de l'image composite RGB
visualization = {
    'min': 0.0,
    'max': 0.3,
    'bands': ['B4', 'B3', 'B2'], # Bande Rouge, Verte, Bleue
}

m = geemap.Map() 
m.add_basemap('HYBRID')
m.set_center(0.85, 8.0, 7)  # Centré géographiquement sur le Togo
m.add_layer(s2, visualization, 'Togo Sentinel-2 RGB (2025)') 
m.add_layer(ee.Image().paint(togo, 0, 2), {'palette': 'black'}, 'Frontières Togo')

In [16]:
m

Map(center=[8.0, 0.85], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', t…